# Etapa 5 — Redes Bayesianas

**DAG com 6 nós:**

```
     PORTE (população)        FRONT (UF faz fronteira com outro país)
              \                 /
               FACCAO (presença de facção, 2024)
             /          |           \
          TXH          ESC          TXT
  (taxa homicídio) (morte a esclarecer) (taxa tentativas)
```

**Justificativa das arestas**
- `PORTE → FACCAO`: cidades maiores concentram mercado consumidor de drogas, logística (portos, rodovias, aeroportos) e presídios, onde as facções se organizam.
- `FRONT → FACCAO`: estados de fronteira estão nas rotas da cocaína vinda da Bolívia, do Peru e da Colômbia.
- `FACCAO → TXH, ESC, TXT`: a disputa por território e as execuções elevam homicídios, tentativas e mortes sem causa esclarecida.

**FACCAO é um colisor** (PORTE → FACCAO ← FRONT), e é ao mesmo tempo a causa comum dos três sintomas. Com essa rede fazemos o raciocínio **diagnóstico**: observamos porte, estado e sintomas e inferimos P(FACCAO | evidências).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 95)


In [2]:
VARS = ["PORTE", "FRONT", "FACCAO", "TXH", "ESC", "TXT"]
PAIS = {"PORTE": [], "FRONT": [], "FACCAO": ["PORTE", "FRONT"],
        "TXH": ["FACCAO"], "ESC": ["FACCAO"], "TXT": ["FACCAO"]}
dados = d[["PORTE", "FRONT", "ALVO", "TXH", "ESC", "TXT"]].rename(columns={"ALVO": "FACCAO"}).astype(int)
VALORES = {v: sorted(dados[v].unique()) for v in VARS}

## 1. CPTs estimadas dos dados (máxima verossimilhança)

In [3]:
CPT = {}
for v in VARS:
    pais = PAIS[v]
    CPT[v] = dados[v].value_counts(normalize=True).sort_index() if not pais else \
             pd.crosstab([dados[p] for p in pais], dados[v], normalize="index")
    print(f"\nP({v}{' | ' + ', '.join(pais) if pais else ''})"); print(CPT[v].round(4))


P(PORTE)
PORTE
0    0.3304
1    0.5384
2    0.1312
Name: proportion, dtype: float64

P(FRONT)
FRONT
0    0.4406
1    0.5594
Name: proportion, dtype: float64

P(FACCAO | PORTE, FRONT)
FACCAO            0       1
PORTE FRONT                
0     0      0.9869  0.0131
      1      0.7807  0.2193
1     0      0.7955  0.2045
      1      0.5792  0.4208
2     0      0.2963  0.7037
      1      0.1392  0.8608

P(TXH | FACCAO)
TXH          0       1       2
FACCAO                        
0       0.3898  0.3279  0.2823
1       0.2162  0.3436  0.4402

P(ESC | FACCAO)
ESC          0       1
FACCAO                
0       0.4080  0.5920
1       0.2046  0.7954

P(TXT | FACCAO)
TXT          0       1      2
FACCAO                       
0       0.3825  0.3115  0.306
1       0.2317  0.3784  0.390


## 2. Economia de parâmetros

In [4]:
card = {v: len(VALORES[v]) for v in VARS}
p_conj = int(np.prod(list(card.values()))) - 1
p_rede = sum((card[v] - 1) * int(np.prod([card[p] for p in PAIS[v]] or [1])) for v in VARS)
print(f"Conjunta completa: {p_conj} parâmetros | Rede: {p_rede} | Economia: {p_conj - p_rede} ({1 - p_rede / p_conj:.0%})")

Conjunta completa: 215 parâmetros | Rede: 19 | Economia: 196 (91%)


## 3. Inferência por enumeração (do zero)

$$P(X \mid e) = \alpha \sum_{y} \prod_i P(x_i \mid pais(X_i))$$

In [5]:
import itertools

def prob(v, valor, atrib):
    pais = PAIS[v]
    if not pais:
        return CPT[v].get(valor, 0.0)
    chave = tuple(atrib[p] for p in pais)
    chave = chave[0] if len(chave) == 1 else chave
    try:
        return CPT[v].loc[chave, valor]
    except KeyError:
        return 0.0

def conjunta(atrib):
    return np.prod([prob(v, atrib[v], atrib) for v in VARS])

def enumeracao(consulta, evidencia):
    ocultas = [v for v in VARS if v != consulta and v not in evidencia]
    dist = {}
    for q in VALORES[consulta]:
        s = 0.0
        for combo in itertools.product(*[VALORES[h] for h in ocultas]):
            atrib = dict(evidencia); atrib[consulta] = q; atrib.update(zip(ocultas, combo))
            s += conjunta(atrib)
        dist[int(q)] = s
    z = sum(dist.values())
    return {k: float(v / z) for k, v in dist.items()}

consultas = [
    ("FACCAO", {}),
    ("FACCAO", {"PORTE": 2}),
    ("FACCAO", {"PORTE": 0}),
    ("FACCAO", {"PORTE": 0, "FRONT": 1, "TXH": 2, "ESC": 1, "TXT": 2}),
    ("FACCAO", {"PORTE": 2, "FRONT": 0, "TXH": 0, "ESC": 0, "TXT": 0}),
    ("FACCAO", {"TXH": 2, "ESC": 1, "TXT": 2}),
    ("TXH", {"FACCAO": 1}),
]
res_enum = {str(e) + q: enumeracao(q, e) for q, e in consultas}
for q, e in consultas:
    print(f"P({q} | {e}) = { {k: round(v, 4) for k, v in res_enum[str(e) + q].items()} }")

P(FACCAO | {}) = {0: 0.6785, 1: 0.3215}
P(FACCAO | {'PORTE': 2}) = {0: 0.2084, 1: 0.7916}
P(FACCAO | {'PORTE': 0}) = {0: 0.8716, 1: 0.1284}
P(FACCAO | {'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, 'TXT': 2}) = {0: 0.5715, 1: 0.4285}
P(FACCAO | {'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, 'TXT': 0}) = {0: 0.7142, 1: 0.2858}
P(FACCAO | {'TXH': 2, 'ESC': 1, 'TXT': 2}) = {0: 0.4415, 1: 0.5585}
P(TXH | {'FACCAO': 1}) = {0: 0.2162, 1: 0.3436, 2: 0.4402}


## 4. Comparação com pgmpy (VariableElimination)

In [6]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
arestas = [(p, v) for v in VARS for p in PAIS[v]]
modelo = DiscreteBayesianNetwork(arestas)
modelo.fit(dados)            # estimador padrão: máxima verossimilhança
ve = VariableElimination(modelo)
linhas = []
for q, e in consultas:
    r = ve.query([q], evidence=e, show_progress=False)
    for i, val in enumerate(r.state_names[q]):
        linhas.append([q, str(e), int(val), res_enum[str(e) + q][int(val)], float(r.values[i])])
cmp = pd.DataFrame(linhas, columns=["consulta", "evidência", "valor", "enumeração (própria)", "pgmpy"])
cmp["diferença"] = (cmp["enumeração (própria)"] - cmp["pgmpy"]).abs()
print("Maior diferença:", cmp["diferença"].max())
cmp.round(5)

Maior diferença: 2.220446049250313e-16


,consulta,evidência,valor,enumeração (própria),pgmpy,diferença
0,FACCAO,{},0,0.67845,0.67845,0.0
1,FACCAO,{},1,0.32155,0.32155,0.0
2,FACCAO,{'PORTE': 2},0,0.20844,0.20844,0.0
3,FACCAO,{'PORTE': 2},1,0.79156,0.79156,0.0
4,FACCAO,{'PORTE': 0},0,0.87156,0.87156,0.0
5,FACCAO,{'PORTE': 0},1,0.12844,0.12844,0.0
6,FACCAO,"{'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, '...",0,0.57150,0.57150,0.0
7,FACCAO,"{'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, '...",1,0.42850,0.42850,0.0
8,FACCAO,"{'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, '...",0,0.71421,0.71421,0.0
9,FACCAO,"{'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, '...",1,0.28579,0.28579,0.0


## 5. d-separação

1. **Causa comum (garfo):** `TXH ← FACCAO → ESC`. Dado FACCAO, os sintomas são independentes entre si.
2. **Colisor:** `PORTE → FACCAO ← FRONT`. Porte e fronteira são independentes **até observarmos FACCAO**. Saber que um município pequeno tem facção torna mais provável que ele esteja num estado de fronteira (*explaining away*).

In [7]:
print("TXH ⊥ ESC    | FACCAO ?   ", not modelo.is_dconnected("TXH", "ESC", observed=["FACCAO"]))
print("PORTE ⊥ FRONT ?           ", not modelo.is_dconnected("PORTE", "FRONT", observed=[]))
print("PORTE ⊥ FRONT | FACCAO ?  ", not modelo.is_dconnected("PORTE", "FRONT", observed=["FACCAO"]))
print()
for p in (0, 2):
    print(f"P(FRONT=1 | PORTE={p})              = {enumeracao('FRONT', {'PORTE': p})[1]:.4f}")
for p in (0, 2):
    print(f"P(FRONT=1 | PORTE={p}, FACCAO=1)    = {enumeracao('FRONT', {'PORTE': p, 'FACCAO': 1})[1]:.4f}")
print("\nConferência nos dados: correlação entre PORTE e FRONT =", round(dados["PORTE"].corr(dados["FRONT"]), 3))

TXH ⊥ ESC    | FACCAO ?    True
PORTE ⊥ FRONT ?            True
PORTE ⊥ FRONT | FACCAO ?   False

P(FRONT=1 | PORTE=0)              = 0.5594
P(FRONT=1 | PORTE=2)              = 0.5594
P(FRONT=1 | PORTE=0, FACCAO=1)    = 0.9552
P(FRONT=1 | PORTE=2, FACCAO=1)    = 0.6083

Conferência nos dados: correlação entre PORTE e FRONT = 0.211


**Leitura.** Sem observar FACCAO, o porte não muda a crença sobre a fronteira (os dois valores são iguais). Quando sabemos que o município **tem facção**, ser **pequeno** aumenta a probabilidade de estar num estado de fronteira: uma facção num município pequeno é melhor "explicada" pela rota do tráfico do que pelo tamanho. A última linha confere a suposição do modelo nos dados: porte e fronteira têm correlação fraca (cerca de 0,2), então a independência entre eles é uma aproximação razoável, mas não exata. Vale citar no relatório.

## 6. A rede como classificador diagnóstico (5-fold)

In [8]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
y = dados["FACCAO"].values
ev_cols = ["PORTE", "FRONT", "TXH", "ESC", "TXT"]
acc, bal, auc = [], [], []
for tr, te in cv.split(dados, y):
    m = DiscreteBayesianNetwork(arestas); m.fit(dados.iloc[tr]); inf = VariableElimination(m)
    cache, probs = {}, []
    for r in dados.iloc[te][ev_cols].values:
        key = tuple(int(v) for v in r)
        if key not in cache:
            q = inf.query(["FACCAO"], evidence=dict(zip(ev_cols, key)), show_progress=False)
            cache[key] = float(q.values[list(q.state_names["FACCAO"]).index(1)])
        probs.append(cache[key])
    probs = np.array(probs); p = (probs >= 0.5).astype(int)
    acc.append(accuracy_score(y[te], p)); bal.append(balanced_accuracy_score(y[te], p)); auc.append(roc_auc_score(y[te], probs))
print(f"Rede bayesiana (limiar 0,5): acurácia={np.mean(acc):.3f}  balanceada={np.mean(bal):.3f}  AUC={np.mean(auc):.3f}")

Rede bayesiana (limiar 0,5): acurácia=0.761  balanceada=0.713  AUC=0.811


In [9]:
combos = dados[ev_cols].drop_duplicates()
mapa = {tuple(int(v) for v in r): enumeracao("FACCAO", dict(zip(ev_cols, map(int, r))))[1] for r in combos.values}
d["p_faccao"] = [mapa[tuple(int(v) for v in r)] for r in dados[ev_cols].values]
d[["uf", "municipio", "cod_ibge", "pop_2022"] + ev_cols + ["variacao_hom", "p_faccao", "ALVO", "REGIME",
   "ALVO_2025", "REGIME_2025", "ENTRADA_2025"]].to_csv("dados/probabilidades_bayesianas.csv", index=False)
print(d["p_faccao"].describe().round(4))

count    808.0000
mean       0.3304
std        0.2890
min        0.0022
25%        0.0800
50%        0.2470
75%        0.5660
max        0.9429
Name: p_faccao, dtype: float64
